# Module 2, Notebook 1: Monte Carlo Prediction

## Learning Objectives
By completing this notebook, you will:
1. Implement first-visit and every-visit Monte Carlo prediction from scratch
2. Apply MC prediction to a simplified Blackjack environment (no external libraries)
3. Plot the value function surface over the Blackjack state space
4. Compare convergence speed of first-visit vs. every-visit methods
5. Understand why MC prediction requires no environment model ($P$ or $R$)

## Prerequisites
- Module 1: policy evaluation, value functions, convergence
- Module 0, Notebook 2: returns and discount factor
- Understanding of episode-based interaction (finite trajectories)

## Estimated Time: 15 minutes

---

**Key Insight:** Monte Carlo prediction estimates $V^\pi(s)$ by asking: "if I follow policy $\pi$ from state $s$, what return do I actually get?" and averaging many such answers. No knowledge of transition probabilities is needed — only the ability to interact with the environment and collect trajectories.

## 1. Setup

In [ ]:
# Purpose: Imports and random seed
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

np.random.seed(42)
print("Ready. NumPy version:", np.__version__)

## 2. Why Monte Carlo? The Model-Free Advantage

Dynamic programming methods (Module 1) require a complete model of the environment: transition probabilities $P(s'|s,a)$ and rewards $R(s,a,s')$. In most real problems, this model is unknown or too expensive to compute.

Monte Carlo methods need only:
- The ability to interact with the environment
- Complete episodes (the episode must terminate)

The estimate of $V(s)$ is simply:
$$\hat{V}(s) = \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_t^{(i)}$$

where $G_t^{(i)}$ is the return observed from the $i$-th visit to state $s$.

By the law of large numbers, $\hat{V}(s) \to V^\pi(s)$ as $N(s) \to \infty$.

## 3. Blackjack Environment

Blackjack is the canonical MC teaching problem (Sutton & Barto, Chapter 5). The state space is small enough to visualize and the game dynamics are simple:

- **State**: (player sum, dealer showing card, player has usable ace)
- **Actions**: HIT (draw card) or STICK (stop)
- **Reward**: +1 win, -1 lose, 0 draw (at terminal state)
- **Episode ends**: player busts (sum > 21), or player sticks (dealer plays out)

We implement this entirely with NumPy — no Gymnasium required.

In [ ]:
# Purpose: Implement the Blackjack environment from scratch
# Key Concept: A model-free environment just needs a step() function; no P or R required

def draw_card():
    """Draw one card. Face cards (J, Q, K) are worth 10; Ace worth 1 or 11."""
    card = np.random.randint(1, 14)  # 1-13
    return min(card, 10)             # J, Q, K all map to 10


def hand_value(cards):
    """
    Compute the best achievable hand value.

    Parameters
    ----------
    cards : list of int

    Returns
    -------
    total : int  — best value (<=21 if possible)
    usable_ace : bool  — True if an ace is counted as 11
    """
    total = sum(cards)
    # Ace is initially counted as 1; promote to 11 if it doesn't bust
    usable_ace = (1 in cards) and (total + 10 <= 21)
    if usable_ace:
        total += 10
    return total, usable_ace


class BlackjackEnv:
    """
    Simplified Blackjack following S&B Chapter 5.

    State: (player_sum, dealer_showing, usable_ace)
      - player_sum:     int in [12, 21]  (we only start from 12; below that we always hit)
      - dealer_showing: int in [1, 10]
      - usable_ace:     bool

    Actions: 0 = STICK, 1 = HIT
    """

    STICK = 0
    HIT   = 1

    def reset(self):
        """
        Deal initial cards and return starting state.

        Player is dealt cards until sum >= 12 (auto-hit on small hands).
        """
        self.player_cards  = [draw_card(), draw_card()]
        self.dealer_cards  = [draw_card(), draw_card()]

        # Auto-hit until player sum >= 12
        while hand_value(self.player_cards)[0] < 12:
            self.player_cards.append(draw_card())

        self.done = False
        return self._get_state()

    def _get_state(self):
        """Return (player_sum, dealer_showing, usable_ace)."""
        player_sum, usable_ace = hand_value(self.player_cards)
        dealer_showing = self.dealer_cards[0]  # Only the face-up card is visible
        return (player_sum, dealer_showing, usable_ace)

    def step(self, action):
        """
        Execute one action.

        Returns
        -------
        next_state : tuple
        reward : float
        done : bool
        """
        if self.done:
            raise RuntimeError("Call reset() before stepping after episode ends.")

        if action == self.HIT:
            self.player_cards.append(draw_card())
            player_sum, usable_ace = hand_value(self.player_cards)

            if player_sum > 21:  # Player busts
                self.done = True
                return self._get_state(), -1.0, True

            return self._get_state(), 0.0, False

        else:  # STICK: dealer plays out
            # Dealer hits until sum >= 17 (fixed dealer policy)
            dealer_sum, _ = hand_value(self.dealer_cards)
            while dealer_sum < 17:
                self.dealer_cards.append(draw_card())
                dealer_sum, _ = hand_value(self.dealer_cards)

            player_sum, _ = hand_value(self.player_cards)

            # Determine winner
            if dealer_sum > 21:        # Dealer busts
                reward = 1.0
            elif player_sum > dealer_sum:
                reward = 1.0
            elif player_sum < dealer_sum:
                reward = -1.0
            else:
                reward = 0.0           # Draw

            self.done = True
            return self._get_state(), reward, True


# Smoke test
env = BlackjackEnv()
state = env.reset()
print(f"Initial state: player_sum={state[0]}, dealer_showing={state[1]}, usable_ace={state[2]}")
state, reward, done = env.step(BlackjackEnv.HIT)
print(f"After HIT:     player_sum={state[0]}, reward={reward}, done={done}")

## 4. Defining a Simple Policy

We evaluate the **stick-on-20+ policy**: hit if player sum < 20, stick otherwise. This is a deterministic threshold policy — simple, not optimal, but easy to visualize.

We generate episodes by running this policy through the Blackjack environment.

In [ ]:
# Purpose: Define the evaluation policy and episode generator
# Key Concept: MC requires complete episodes; partial trajectories cannot be used

def policy_stick_on_20(state):
    """
    Stick if player sum >= 20, hit otherwise.
    Ignores dealer card and ace status (simple threshold policy).
    """
    player_sum, _, _ = state
    return BlackjackEnv.STICK if player_sum >= 20 else BlackjackEnv.HIT


def generate_episode(env, policy_fn, max_steps=30):
    """
    Run one complete episode and return the trajectory.

    Parameters
    ----------
    env : BlackjackEnv
    policy_fn : callable
        Maps state -> action.
    max_steps : int
        Safety cap against infinite loops.

    Returns
    -------
    trajectory : list of (state, action, reward)
    """
    state = env.reset()
    trajectory = []

    for _ in range(max_steps):
        action = policy_fn(state)
        next_state, reward, done = env.step(action)
        trajectory.append((state, action, reward))
        state = next_state
        if done:
            break

    return trajectory


# Generate a few example episodes
env = BlackjackEnv()
for i in range(3):
    np.random.seed(i)
    ep = generate_episode(env, policy_stick_on_20)
    total_reward = sum(r for _, _, r in ep)
    print(f"Episode {i+1}: {len(ep)} steps, final reward={total_reward:+.0f}")
    for t, (s, a, r) in enumerate(ep):
        action_name = 'STICK' if a == 0 else 'HIT'
        print(f"  t={t}: state={s}, action={action_name}, reward={r:+.1f}")

## 5. First-Visit and Every-Visit MC Prediction

Both methods estimate $V^\pi(s)$ by averaging observed returns:

- **First-visit**: for each episode, count the return $G_t$ only at the *first* time state $s$ is visited
- **Every-visit**: count the return $G_t$ at *every* time $s$ is visited

Both converge to $V^\pi(s)$ as $N \to \infty$. First-visit has lower bias (unbiased); every-visit has lower variance (uses more data). For Blackjack, episodes rarely revisit states, so the difference is small.

In [ ]:
# Purpose: Implement first-visit and every-visit MC prediction
# Key Concept: The only difference is whether we track the first visit or all visits

def compute_episode_returns(trajectory, gamma=1.0):
    """
    Compute the discounted return G_t for each timestep in the trajectory.

    Parameters
    ----------
    trajectory : list of (state, action, reward)
    gamma : float

    Returns
    -------
    returns : list of float
        G_t for t = 0, 1, ..., T-1
    """
    rewards = [r for _, _, r in trajectory]
    T = len(rewards)
    returns = [0.0] * T
    G = 0.0
    for t in range(T - 1, -1, -1):
        G = rewards[t] + gamma * G
        returns[t] = G
    return returns


def mc_prediction(env, policy_fn, n_episodes, gamma=1.0, method='first_visit'):
    """
    Monte Carlo prediction: estimate V^pi from sampled episodes.

    Parameters
    ----------
    env : BlackjackEnv
    policy_fn : callable
    n_episodes : int
    gamma : float
    method : str
        'first_visit' or 'every_visit'

    Returns
    -------
    V : dict mapping state -> estimated value
    visit_counts : dict mapping state -> total number of visits
    v_over_time : dict mapping state -> list of V estimates after each update
        (only for a few tracked states to keep memory manageable)
    """
    returns_sum   = {}  # state -> sum of observed returns
    visit_counts  = {}  # state -> number of returns observed

    # Track convergence for a few representative states
    TRACKED_STATES = [(20, 1, False), (18, 5, False), (15, 7, False)]
    v_over_time = {s: [] for s in TRACKED_STATES}

    for episode_idx in range(n_episodes):
        trajectory = generate_episode(env, policy_fn)
        returns    = compute_episode_returns(trajectory, gamma)

        # First-visit: record only first occurrence of each state in this episode
        if method == 'first_visit':
            first_visit_t = {}  # state -> first timestep it appears
            for t, (state, _, _) in enumerate(trajectory):
                if state not in first_visit_t:
                    first_visit_t[state] = t

            for state, t_first in first_visit_t.items():
                G = returns[t_first]
                returns_sum[state]  = returns_sum.get(state, 0.0) + G
                visit_counts[state] = visit_counts.get(state, 0)  + 1

        # Every-visit: record return at every occurrence of each state
        elif method == 'every_visit':
            for t, (state, _, _) in enumerate(trajectory):
                G = returns[t]
                returns_sum[state]  = returns_sum.get(state, 0.0) + G
                visit_counts[state] = visit_counts.get(state, 0)  + 1

        else:
            raise ValueError(f"Unknown method: {method}. Use 'first_visit' or 'every_visit'.")

        # Update convergence tracking
        for s_track in TRACKED_STATES:
            if s_track in returns_sum:
                v_est = returns_sum[s_track] / visit_counts[s_track]
                v_over_time[s_track].append(v_est)
            else:
                v_over_time[s_track].append(float('nan'))

    # Compute final value estimates
    V = {state: returns_sum[state] / visit_counts[state]
         for state in returns_sum}

    return V, visit_counts, v_over_time


N_EPISODES = 50_000
env = BlackjackEnv()

print(f"Running first-visit MC prediction ({N_EPISODES:,} episodes)...")
V_fv, counts_fv, v_time_fv = mc_prediction(
    env, policy_stick_on_20, N_EPISODES, gamma=1.0, method='first_visit'
)

print(f"Running every-visit MC prediction ({N_EPISODES:,} episodes)...")
V_ev, counts_ev, v_time_ev = mc_prediction(
    env, policy_stick_on_20, N_EPISODES, gamma=1.0, method='every_visit'
)

print(f"\nStates visited: {len(V_fv)} (first-visit), {len(V_ev)} (every-visit)")
print("\nSpot checks (player_sum, dealer_showing, usable_ace):")
for state in [(20, 1, False), (18, 5, False), (15, 7, False)]:
    v_fv = V_fv.get(state, float('nan'))
    v_ev = V_ev.get(state, float('nan'))
    print(f"  {state}: first-visit={v_fv:.4f}, every-visit={v_ev:.4f}")

## 6. Value Function Surface Plot

The Blackjack value function is best visualized as a 3D surface over (player_sum, dealer_showing) for fixed usable_ace. States closer to 21 have higher value; states where the dealer shows high cards are less favorable.

In [ ]:
# Purpose: Visualize V as a 3D surface over the Blackjack state space
# Key Concept: 3D surfaces reveal the value landscape in a compact, human-interpretable form

def plot_blackjack_value_surface(V, title, usable_ace=False):
    """
    Plot V(player_sum, dealer_showing) as a 3D surface.

    Parameters
    ----------
    V : dict  — state -> value
    title : str
    usable_ace : bool

    Returns
    -------
    fig : matplotlib Figure
    """
    player_sums     = np.arange(12, 22)   # 12-21
    dealer_showings = np.arange(1, 11)    # Ace-10

    # Build value grid
    Z = np.zeros((len(player_sums), len(dealer_showings)))
    for i, ps in enumerate(player_sums):
        for j, ds in enumerate(dealer_showings):
            state = (ps, ds, usable_ace)
            Z[i, j] = V.get(state, 0.0)

    X, Y = np.meshgrid(dealer_showings, player_sums)

    fig = plt.figure(figsize=(10, 7))
    ax  = fig.add_subplot(111, projection='3d')

    surf = ax.plot_surface(X, Y, Z, cmap='RdYlGn', edgecolor='none', alpha=0.9)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='V(s)')

    ax.set_xlabel('Dealer showing', fontsize=11)
    ax.set_ylabel('Player sum', fontsize=11)
    ax.set_zlabel('V(s)', fontsize=11)
    ax.set_title(f'{title}\n(usable ace={usable_ace})', fontsize=12, fontweight='bold')
    ax.set_xticks(dealer_showings)
    ax.set_xticklabels(['A' if d == 1 else str(d) for d in dealer_showings], fontsize=8)
    ax.set_yticks(player_sums)
    ax.view_init(elev=30, azim=-120)

    return fig


# Plot for no usable ace (the more common case)
fig = plot_blackjack_value_surface(
    V_fv,
    title=f'First-Visit MC — V^pi (stick-on-20+, {N_EPISODES:,} episodes)',
    usable_ace=False
)
plt.savefig('blackjack_value_surface.png', dpi=120, bbox_inches='tight')
plt.show()
print("Surface plot saved to blackjack_value_surface.png")

## 7. Convergence Comparison

We track how the value estimate at a few states evolves as more episodes are collected. Both methods should converge to similar values; every-visit should get there faster for frequently visited states.

In [ ]:
# Purpose: Compare convergence speed of first-visit vs every-visit MC
# Key Concept: Every-visit uses more data per episode, so it converges faster but with higher bias

TRACKED_STATES = [(20, 1, False), (18, 5, False), (15, 7, False)]
state_labels   = ['(sum=20, dealer=A)', '(sum=18, dealer=5)', '(sum=15, dealer=7)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, state, label in zip(axes, TRACKED_STATES, state_labels):
    fv_trace = np.array(v_time_fv[state])
    ev_trace = np.array(v_time_ev[state])

    # Smooth traces for readability
    window = 500
    fv_valid = ~np.isnan(fv_trace)
    ev_valid = ~np.isnan(ev_trace)

    x_fv = np.where(fv_valid)[0]
    x_ev = np.where(ev_valid)[0]

    ax.plot(x_fv, fv_trace[fv_valid], color='#1c7ed6', linewidth=1.2,
            alpha=0.5, label='First-visit')
    ax.plot(x_ev, ev_trace[ev_valid], color='#e64980', linewidth=1.2,
            alpha=0.5, label='Every-visit')

    # Smoothed versions
    if len(fv_trace[fv_valid]) > window:
        smooth_fv = np.convolve(fv_trace[fv_valid], np.ones(window)/window, mode='valid')
        ax.plot(x_fv[window-1:], smooth_fv, color='#1c7ed6', linewidth=2.5)
    if len(ev_trace[ev_valid]) > window:
        smooth_ev = np.convolve(ev_trace[ev_valid], np.ones(window)/window, mode='valid')
        ax.plot(x_ev[window-1:], smooth_ev, color='#e64980', linewidth=2.5)

    final_fv = fv_trace[fv_valid][-1] if fv_valid.any() else float('nan')
    final_ev = ev_trace[ev_valid][-1] if ev_valid.any() else float('nan')
    ax.axhline(final_fv, color='#1c7ed6', linestyle='--', linewidth=1, alpha=0.7)

    ax.set_xlabel('Episode', fontsize=11)
    ax.set_ylabel('V(s) estimate', fontsize=11)
    ax.set_title(f'State {label}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('First-Visit vs Every-Visit MC Convergence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mc_convergence_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to mc_convergence_comparison.png")

## 8. Exercises

### Exercise 1.1: Evaluate a Different Policy

**Task:** Define a **conservative policy** that sticks if player sum >= 17 (instead of 20). Run first-visit MC prediction with 50,000 episodes. Compare V at state (17, 5, False) under the two policies (stick-on-20 vs stick-on-17). Which policy gets a higher value at that state? Why does this make sense?

**Expected Output:** Two floats, V(17,5,False) under each policy, with a brief printed explanation.

**Hints:**
<details>
<summary>Hint 1</summary>
Copy `policy_stick_on_20` and change the threshold from 20 to 17.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
A policy that sticks early (on 17) risks fewer busts but may lose to the dealer more often. The value at any state depends on the policy you're following, not just the state itself.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Step 1: Define policy_stick_on_17
def policy_stick_on_17(state):
    """Stick if player sum >= 17, hit otherwise."""
    # YOUR IMPLEMENTATION HERE
    return None  # Remove this line


# Step 2: Run MC prediction with the new policy
V_conservative = None  # dict mapping state -> value

# Step 3: Extract V at state (17, 5, False) for both policies
TARGET_STATE = (17, 5, False)
v_stick20_target = None
v_stick17_target = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_1():
    # Test policy function
    assert policy_stick_on_17((17, 5, False)) == BlackjackEnv.STICK, \
        "policy_stick_on_17 should STICK when player sum=17."
    assert policy_stick_on_17((16, 5, False)) == BlackjackEnv.HIT, \
        "policy_stick_on_17 should HIT when player sum=16."
    assert policy_stick_on_17((21, 1, True)) == BlackjackEnv.STICK, \
        "policy_stick_on_17 should STICK when player sum=21."

    assert V_conservative is not None, "Run mc_prediction to get V_conservative!"
    assert isinstance(V_conservative, dict), "V_conservative should be a dict."
    assert len(V_conservative) > 50, "V_conservative should have many states estimated."

    assert v_stick17_target is not None, "Extract v_stick17_target!"
    assert v_stick20_target is not None, "Extract v_stick20_target from V_fv!"

    # Both should be floats in [-1, 1]
    for name, val in [('stick17', v_stick17_target), ('stick20', v_stick20_target)]:
        assert isinstance(val, float), f"V({name}) should be a float."
        assert -1.0 <= val <= 1.0, f"V({name}) must be in [-1, 1], got {val:.3f}"

    print(f"Exercise 1.1 passed!")
    print(f"  V{TARGET_STATE} under stick-on-20: {v_stick20_target:.4f}")
    print(f"  V{TARGET_STATE} under stick-on-17: {v_stick17_target:.4f}")
    higher = 'stick-on-17' if v_stick17_target > v_stick20_target else 'stick-on-20'
    print(f"  Higher value at this state: {higher}")

test_exercise_1_1()

### Exercise 1.2: Sample Efficiency — Episodes Needed for Accuracy

**Task:** Run first-visit MC prediction with episode counts $N \in \{100, 1000, 10000, 50000\}$. For each $N$, compute the estimated V at state (18, 5, False). Plot a convergence curve: V estimate vs. log(N). How many episodes does it take for the estimate to stabilize within 0.05 of the 50,000-episode estimate?

**Expected Output:** A matplotlib plot and the episode count where stabilization occurs.

**Hints:**
<details>
<summary>Hint 1</summary>
Call `mc_prediction` four times with increasing `n_episodes`. The V at state (18, 5, False) is `V.get((18, 5, False), float('nan'))`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
N_VALUES   = [100, 1_000, 10_000, 50_000]
EVAL_STATE = (18, 5, False)

v_estimates = []  # One float per N in N_VALUES
for n in N_VALUES:
    pass  # Replace with: run mc_prediction and extract V[EVAL_STATE]

# Plot: V estimate vs. N (log scale on x-axis)
# fig, ax = plt.subplots(...)  # Your plot here

# Find N where |V_estimate - V_50000| < 0.05
stabilization_N = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_2():
    assert len(v_estimates) == len(N_VALUES), \
        f"Need {len(N_VALUES)} estimates, one per N. Got {len(v_estimates)}."

    for i, (n, v) in enumerate(zip(N_VALUES, v_estimates)):
        assert not np.isnan(v), f"V estimate for N={n} is NaN — state not visited?"
        assert -1.0 <= v <= 1.0, f"V estimate for N={n} must be in [-1,1], got {v:.3f}"

    # Estimate should converge: variance should decrease with N
    # Check that the 50000-episode estimate is available
    v_large = v_estimates[-1]  # N=50000
    assert v_large is not None

    assert stabilization_N is not None, "Compute stabilization_N!"
    assert stabilization_N in N_VALUES, \
        f"stabilization_N should be one of {N_VALUES}, got {stabilization_N}"

    print(f"Exercise 1.2 passed!")
    print(f"  V estimates for {EVAL_STATE}:")
    for n, v in zip(N_VALUES, v_estimates):
        print(f"    N={n:>6,}: V = {v:.4f}")
    print(f"  Stabilization (within 0.05 of N=50000): N >= {stabilization_N}")

test_exercise_1_2()

## Key Takeaways

1. **Monte Carlo prediction is model-free**: it estimates $V^\pi$ purely from sampled returns, with no knowledge of $P$ or $R$. This is the key advantage over DP.
2. **First-visit MC** records the return only at the first occurrence of a state per episode. **Every-visit MC** records at every occurrence. Both converge to $V^\pi$ but with different bias-variance trade-offs.
3. **Complete episodes are required**: MC cannot update values mid-episode. This limits MC to episodic tasks (tasks that eventually terminate).
4. **Sample efficiency matters**: rare states require many episodes to get an accurate estimate. State (12, 1, True) may appear in very few episodes.
5. **The value function surface** reveals the structure of the game: high values near 21, low values when dealer shows a strong card.
6. **The policy being evaluated matters**: V is always defined relative to a specific policy. The same state can have very different values under different policies.

## What's Next

In Notebook 2 (`02_mc_control.ipynb`), we go from prediction (estimating $V^\pi$ for a fixed $\pi$) to control (finding the optimal $\pi^*$). MC control combines Q-value estimation with policy improvement, but it needs exploration to work — which is where the epsilon-soft policy comes in.